In [ ]:
import os
import zipfile
import pickle
import pandas as pd
from tqdm import tqdm  # Import tqdm for the progress bar

def load_dataframe_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Iterate through all files in the zip
        for file_name in zip_ref.namelist():
            if file_name.endswith('.pkl'):
                with zip_ref.open(file_name) as file:
                    # Load the data from the pickle file
                    data = pickle.load(file)
                    
                    # Check if the data is a list, and if so, convert it to a DataFrame
                    if isinstance(data, list):
                        return pd.DataFrame(data)
                    else:
                        print(f"Warning: The file {file_name} does not contain a list or DataFrame.")
    return None

def combine_pickles_in_folders(folders):
    combined_df = pd.DataFrame()
    
    # Loop through each folder
    for folder in folders:
        for root, dirs, files in os.walk(folder):
            # Add tqdm to show progress when iterating over the files
            for file in tqdm(files, desc=f"Processing files in {folder}", unit="file"):
                if file.endswith('.zip'):
                    zip_path = os.path.join(root, file)
                    # print(f"Processing {zip_path}")
                    
                    # Load the dataframe from the zip file
                    df = load_dataframe_from_zip(zip_path)
                    if df is not None:
                        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df

# Define the folders to search
folders = ['processed_data', 'processed_data_new']

# Combine the dataframes
combined_dataframe = combine_pickles_in_folders(folders)

# Save the combined dataframe as a Parquet file
if not combined_dataframe.empty:
    combined_dataframe.to_parquet('combined_dataframe.parquet', compression='gzip')
    print("DataFrames combined and saved as 'combined_dataframe.parquet'")
else:
    print("No data found to combine.")


Processing files in processed_data:   3%|█▏                                          | 7/266 [00:14<07:52,  1.83s/file]